In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Monte Carlo Simulation for Project Risk
#
# This notebook demonstrates how Monte Carlo simulation can be
# used to quantify project cost uncertainty and support
# risk-aware decision-making.
#
# The model simulates thousands of possible cost outcomes based
# on minimum, most likely, and maximum estimates with associated
# likelihoods, and reports P50 and P90 risk metrics.
# ============================================================


# -------------------------------------------
# Load risk data
# -------------------------------------------

df = pd.read_csv("data/risk_distributions_sample.csv")

# --- Validate input data ---
required_cols = ["Likelihood", "Min", "Most Likely", "Max"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")
    df[col] = pd.to_numeric(df[col], errors="coerce")

if df[required_cols].isna().any().any():
    raise ValueError("Non-numeric or NaN values detected.")

if not ((0 <= df["Likelihood"]) & (df["Likelihood"] <= 1)).all():
    raise ValueError("Likelihood values must be between 0 and 1.")

if not (df["Min"] <= df["Most Likely"]).all() or not (df["Most Likely"] <= df["Max"]).all():
    raise ValueError("Each row must satisfy Min ≤ Most Likely ≤ Max.")


# -------------------------------------------
# Vectorized Monte Carlo Simulation
# -------------------------------------------

def simulate_totals(df, trials=50_000, seed=42):
    rng = np.random.default_rng(seed)
    totals = np.zeros(trials)

    a = df["Min"].to_numpy()
    m = df["Most Likely"].to_numpy()
    b = df["Max"].to_numpy()
    p = df["Likelihood"].to_numpy()

    for ai, mi, bi, pi in zip(a, m, b, p):
        occurs = rng.random(trials) < pi
        draws = rng.triangular(ai, mi, bi, size=trials)
        totals += draws * occurs.astype(float)

    return totals


results = simulate_totals(df, trials=50_000, seed=42)

# -------------------------------------------
# Summary statistics
# -------------------------------------------

p50 = float(np.percentile(results, 50))
p90 = float(np.percentile(results, 90))
mean = float(np.mean(results))
std = float(np.std(results, ddof=1))

print(f"P50 ≈ ${p50:,.0f}")
print(f"P90 ≈ ${p90:,.0f}")
print(f"Mean ≈ ${mean:,.0f}")
print(f"Standard Deviation ≈ ${std:,.0f}")


# -------------------------------------------
# Visualization
# -------------------------------------------

q75, q25 = np.percentile(results, [75, 25])
iqr = q75 - q25
bin_width = 2 * iqr * (len(results) ** (-1 / 3))
bins = max(30, int((results.max() - results.min()) / bin_width)) if bin_width > 0 else 60

plt.figure(figsize=(9, 5.5))
plt.hist(results, bins=bins)

plt.axvline(p50, linestyle="--", linewidth=2)
plt.axvline(p90, linestyle="--", linewidth=2)

plt.title("Total Project Risk Cost — Monte Carlo Simulation (50,000 runs)")
plt.xlabel("Total risk cost (USD)")
plt.ylabel("Frequency")

ymax = plt.gca().get_ylim()[1]
plt.text(p50, ymax * 0.92, f"P50 ≈ ${p50:,.0f}", rotation=90, va="top", ha="right")
plt.text(p90, ymax * 0.92, f"P90 ≈ ${p90:,.0f}", rotation=90, va="top", ha="left")

plt.tight_layout()
plt.show()


# -------------------------------------------
# Decision Insight
# -------------------------------------------

print(
    f"Recommendation: Set contingency close to P90 ≈ ${p90:,.0f}, "
    f"which is about ${(p90 - p50):,.0f} above the P50 estimate."
)
